# Asch in Silicon — experiment notebook

[repo](https://github.com/LihanCanCode/Collective-Cognitive-Error)

### Before you run

Kaggle sidebar → **Settings**: **Accelerator** `GPU T4 x2`, **Internet** `On`.

---

## 🚦 Fresh session: which cells to run

**To produce the core results table (the deliverable), run only these four:**

| # | Cell | Why | ~time |
|---|---|---|---|
| 1 | **Cell 1** — environment check | confirms the GPU is actually attached | 10 s |
| 2 | **Cell 2** — clone | pulls the latest code; defines `REPO_DIR` | 30 s |
| 3 | **Cell 3** — mock verification | full pipeline offline; if this fails the harness is broken and no GPU time should be spent | 30 s |
| 4 | **Cell 10** — ⭐ core results sweep | **the deliverable** | **~4-5h/model, sequential** |

**Skip** Cells 4–7 (gate + inspection), Optional A/B, Cell 8, Cell 9 — all diagnostics from
earlier rounds, already passed.

### ⚠️ Cell 10 is sequential by design, and that's slow on purpose

Cell 8 found real (small but nonzero) answer-level divergence between batched and sequential
generation — GPU batched matmul uses a different reduction order than single-example inference,
which can occasionally flip a near-tied greedy decision. At the single-digit-percent effect sizes
here, that is enough to move a reported number. **So Cell 10 defaults to `--batch-size 1`.**
Sequential, resumable, split across sessions freely — do not "fix" the slowness by adding
`--batch-size 16` to Cell 10 itself. If you want a fast directional look, use **Optional C**
further down, which writes to a clearly separate `_fastlook` directory precisely so its numbers
can never be mistaken for the ones that go in the paper.

## Cell 1 — check the environment

Nothing to install. The gate runs on **`transformers`**, which Kaggle already ships.

That is deliberate: the gate is only ~250 generations, so vLLM's throughput buys nothing, while a
pinned vLLM hard-fails on any model config newer than itself — `vllm==0.6.3` cannot parse Qwen2.5's
`rope_scaling` and dies on a bare `AssertionError`. vLLM returns for the full grid, where
throughput is the entire point.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch, transformers

print("transformers", transformers.__version__, "| torch", torch.__version__)
print("CUDA devices:", torch.cuda.device_count())
assert torch.cuda.is_available(), "No GPU. Set Accelerator to 'GPU T4 x2' in the sidebar."

## Cell 2 — get the code and verify the clone

Every path below is absolute and anchored to `REPO_DIR`, so nothing depends on the notebook's
working directory. The assertion is there because a partial or failed clone otherwise shows up
several cells later as a confusing import error.

> ⚠️ **This cell runs `rm -rf` on `REPO_DIR`.** Do not re-run it after producing results you care
> about — it deletes anything written inside the repo. Cell 10 writes to
> `/kaggle/working/results_arms` for exactly this reason.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/LihanCanCode/Collective-Cognitive-Error.git"
REPO_DIR = Path("/kaggle/working/repo")

!rm -rf {REPO_DIR}
!git clone -q {REPO_URL} {REPO_DIR}

expected = ["scripts/run_smoke.py", "scripts/make_smoke_bank.py", "src/asch/runner.py"]
missing = [p for p in expected if not (REPO_DIR / p).exists()]
assert not missing, f"clone incomplete, missing: {missing}"

print("clone OK")
print(subprocess.run(["git", "log", "--oneline", "-1"], cwd=REPO_DIR,
                     capture_output=True, text=True).stdout.strip())

## Cell 3 — verify the pipeline offline

A full end-to-end run on the mock backend: no GPU, no network, no model. If this does not print
`PASS`, the harness itself is broken and no GPU time should be spent on it.

In [ ]:
!python {REPO_DIR}/scripts/make_smoke_bank.py
!python {REPO_DIR}/scripts/run_smoke.py --backend mock --out {REPO_DIR}/results/mock_check.jsonl

## Cell 4 — run the gate

50 items × {n=0 control, n=3 unanimous-wrong} = 100 trials, ~250 generations. n=3 is where human
conformity peaks in Asch (32%).

`device_map="auto"` shards the 7B across both T4s — a 7B in fp16 is ~15 GB and will not fit on one.
Expect **15–30 min**, most of it the model download.

Resumable: if the session dies, just re-run this cell — completed trials are skipped.

In [ ]:
MODEL = "Qwen/Qwen2.5-7B-Instruct"
RESULTS = REPO_DIR / "results" / "smoke_qwen7b.jsonl"

!python {REPO_DIR}/scripts/run_smoke.py --backend hf --model {MODEL} --out {RESULTS}

## Cell 5 — read five transcripts by hand

Not optional. You are checking three things:
1. Did each confederate actually assert its assigned answer? (`complied=True`)
2. Did the naive agent answer in the `Answer: X` format? (`valid=True`)
3. Does the naive agent's reasoning look like real deliberation, or is it degenerate?

In [ ]:
import json

assert RESULTS.exists(), (
    f"{RESULTS} not found — Cell 4 did not produce results. Scroll up and read its error "
    "output before continuing."
)

records = [json.loads(line) for line in RESULTS.open() if line.strip()]
critical = [r for r in records if r["n_confederates"] == 3]
print(f"{len(records)} trials total, {len(critical)} critical\n")

for rec in critical[:5]:
    print("=" * 95)
    print(
        f"stance={rec['stance']}  answer={rec['answer']}  correct={rec['correct_answer']}  "
        f"majority={rec['majority_answer']}  valid={rec['valid']}"
    )
    for turn in rec.get("transcript", []):
        who = turn["role"]
        extra = (
            f" (assigned {turn['assigned_answer']}, complied={turn['complied']})"
            if who == "confederate"
            else ""
        )
        print(f"\n--- {who}{extra} ---")
        print(turn["text"][:400])

## Cell 6 — diagnostics

Full breakdown: **per-subtype baseline accuracy** (which items are too hard) and the **discard
split** (confederate character-breaks vs parse failures — different problems, different fixes).

Runs on the saved JSONL in seconds, no GPU. **Report this output back.**

In [ ]:
!python {REPO_DIR}/scripts/diagnose.py {RESULTS}

## Cell 7 — save results off the session

Kaggle wipes everything outside `/kaggle/working` when the session ends. Either **Save Version**
(persists `/kaggle/working` as notebook output) or download the JSONL from the file browser.

In [ ]:
import shutil

dest = Path("/kaggle/working") / RESULTS.name
shutil.copy(RESULTS, dest)
print(f"saved -> {dest}  ({dest.stat().st_size / 1024:.0f} KB)")

## Optional A — the three-arm contrast (**the paper's central measurement**)

Pilot result on Qwen2.5-7B, identical bank: **JUSTIFIED 16.0%** vs **BARE 2.0%**.

Asch's confederates were bare — they stated a line and said nothing else — and his humans still
conformed at 32%. A model that conforms at 2% to the same thing is not socially conformist. It
shifts only when confederates supply an *argument*, and those arguments are fabricated.

**But bare turns are also shorter**, so the gap could be textual salience rather than
argumentation. `FILLER` settles it: the answer plus a content-free sentence of comparable length —
same visual weight, no argument.

| If FILLER ≈ BARE | If FILLER ≈ JUSTIFIED |
|---|---|
| argumentation drives it — the headline claim holds | mere presence of text drives it — reframe |

`bare` and `filler` need **no confederate generation at all**, so both arms are nearly free.

In [ ]:
import json
from pathlib import Path

ARMS = ["bare", "filler", "justified"]
arm_paths = {a: REPO_DIR / "results" / f"smoke_arm_{a}.jsonl" for a in ARMS}

for arm, path in arm_paths.items():
    print(f"\n{'=' * 30} {arm.upper()} {'=' * 30}")
    !python {REPO_DIR}/scripts/run_smoke.py --backend hf --model {MODEL} \
        --confederate-style {arm} --out {path}

# Chain-of-thought arm: same confederates, but the model must commit BEFORE it may reason.
# Session-3 hypothesis: forcing explicit reasoning before committing suppresses conformity.
AF = REPO_DIR / "results" / "smoke_arm_justified_answerfirst.jsonl"
print(f"\n{'=' * 25} JUSTIFIED / ANSWER-FIRST {'=' * 25}")
!python {REPO_DIR}/scripts/run_smoke.py --backend hf --model {MODEL} \
    --confederate-style justified --response-format answer_first --out {AF}
arm_paths["justified+answer_first"] = AF

# --- the comparison table -------------------------------------------------------------
print("\n\n" + "=" * 74)
print(f"{'arm':<26} {'baseline err':>13} {'conformity':>12} {'n':>6}")
print("=" * 74)

summary = {}
for arm, path in arm_paths.items():
    recs = [json.loads(l) for l in path.open() if l.strip()]
    ctrl = [r for r in recs if r["n_confederates"] == 0 and r["valid"]]
    crit = [r for r in recs if r["n_confederates"] == 3 and r["valid"]]
    base = sum(1 for r in ctrl if r["answer"] != r["correct_answer"]) / max(len(ctrl), 1)
    cr = sum(1 for r in crit if r["stance"] == "adopted") / max(len(crit), 1)
    summary[arm] = (base, cr)
    print(f"{arm:<26} {base:>12.1%} {cr:>11.1%} {len(crit):>6}")

print("=" * 74)
if abs(summary["filler"][1] - summary["bare"][1]) < abs(summary["filler"][1] - summary["justified"][1]):
    print("FILLER tracks BARE -> ARGUMENTATION drives the effect. Headline claim holds.")
else:
    print("FILLER tracks JUSTIFIED -> mere presence of text drives it. Reframe needed.")

af_base, af_cr = summary["justified+answer_first"]
rf_base, rf_cr = summary["justified"]
print(f"\nCoT check: reasoning_first {rf_cr:.1%} vs answer_first {af_cr:.1%} conformity")
print(f"           baseline error   {rf_base:.1%} vs {af_base:.1%}")
print("If answer_first is both less accurate AND more conformist, deliberation is a defence.")

## Optional B — a second model

If Qwen-7B fails the gate, the fastest diagnostic is a second model: a model-specific quirk looks
very different from a genuine ceiling in the item bank. All ungated, no HF token needed.

Re-run Cells 5–7 afterwards to inspect the new results.

In [ ]:
# mistralai/Mistral-7B-Instruct-v0.3
# google/gemma-2-9b-it
# Qwen/Qwen2.5-1.5B-Instruct    <- smaller and fast: expect MORE conformity if the effect is real

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
RESULTS = REPO_DIR / "results" / "smoke_qwen1_5b.jsonl"

!python {REPO_DIR}/scripts/run_smoke.py --backend hf --model {MODEL} --out {RESULTS}

---

## Cell 8 — verify batching **on real hardware**

The batched path is tested against the sequential path in `pytest`, but only through the mock
backend — which cannot catch a real-GPU numerical effect.

Run the same 100 trials batched and diff against the sequential results. Two kinds of
disagreement are possible and they mean very different things:

* **answer-level** — the model reached a different conclusion. This is the only kind that matters.
* **text-only** — same answer, different phrasing en route to it. Expected and harmless; a naive
  string diff would flag this as a "mismatch" even though nothing measurable changed.

Known cause of real (rare) answer-level flips: batched GPU matmul uses a different reduction order
than a single-example forward pass, which can occasionally flip a near-tied greedy argmax — even
at temperature 0, even with correct left-padding (verified). This is standard GPU batching
numerics, not a masking bug — but it still matters here, because these effect sizes are small
enough that a few flipped trials can move a headline number.

In [ ]:
SEQ = REPO_DIR / "results" / "smoke_qwen7b.jsonl"            # from Cell 4 (sequential)
BATCHED = REPO_DIR / "results" / "smoke_qwen7b_batched.jsonl"

import time
started = time.time()
!python {REPO_DIR}/scripts/run_smoke.py --backend hf --model Qwen/Qwen2.5-7B-Instruct \
    --batch-size 16 --out {BATCHED}
print(f"\nbatched run: {(time.time() - started) / 60:.1f} min")

!python {REPO_DIR}/scripts/compare_runs.py {SEQ} {BATCHED}

## Cell 9 — calibration pre-pass

The step that makes conformity *attributable*. Asks every item alone, 5 samples at temperature
0.7, and keeps only those the model gets right ≥95% of the time (i.e. 5/5).

Sampling above temperature 0 is deliberate: an item the model only gets right under greedy
decoding is not one it **knows**, and that fragility would resurface later as spurious
"conformity".

Output is a **per-model** bank — a bank calibrated for Qwen-7B is not valid for Llama-8B, and
cross-model comparisons use the intersection.

In [ ]:
!python {REPO_DIR}/scripts/calibrate.py \
    --backend hf --model Qwen/Qwen2.5-7B-Instruct \
    --samples 5 --batch-size 16

# Read the per-subtype table carefully. Gate run 2 showed conformity ranging from 0% (list_count)
# to 35% (magnitude), so if a subtype loses most of its items here, the surviving bank is skewed
# toward whichever subtypes calibrate cleanly — and that alone moves the headline number.

---

# ⭐ Cell 10 — THE CORE RESULTS TABLE (run this on every model)

One command produces the paper's central argument. Each row removes one candidate artefact:

| confederates | format | tests | Qwen-7B pilot |
|---|---|---|---|
| justified | answer_first | **≈ how prior work measures it** | 20.0% |
| justified | reasoning_first | + allow deliberation | 6.0% |
| filler | reasoning_first | + remove the argument, keep the text | 0.0% |
| bare | reasoning_first | + Asch's actual paradigm | 0.0% |
| bare | answer_first | completes the 2×2 | — |

**The top row matters most.** A null result alone reads as *"you failed to find the effect."*
Reproducing the literature's magnitude and *then* dissolving it is the contribution.

**Uses the 200-item main bank, not the 50-item gate bank.** At n=50 the combined effect was
significant (p=0.0012) but the individual artefacts were not (p=0.07, p=0.24) — underpowered, not
null. 129 items/arm gives 80% power.

> 🚨 **Runs SEQUENTIALLY (`--batch-size 1`) by default — this is deliberate, not an oversight.**
> Cell 8 found real answer-level divergence between batched and sequential generation. At the
> fragile margins we first measured, that mattered a lot; on the 200-item confirmation run the
> effects turned out to be much larger (tens of percentage points), so batching noise is unlikely
> to overturn the qualitative story — but the *exact numbers* reported in a paper table should
> still come from the sequential run. ~4-5h/model on a T4; resumable, split across sessions freely.
>
> A batched pass is available further down as an *explicitly separate, exploratory-only* cell —
> useful for an early look, never for a number you plan to report.

**⚠️ Gemma is gated and will 401.** Swapped the third model to `microsoft/Phi-3.5-mini-instruct`
(ungated). See the note on the code cell below for how to use Gemma instead if you get a token.

**⚠️ If a model's baseline error is well above 0%** (this bank is calibrated for nothing in
particular — it happened to be trivial for Qwen but was NOT for Mistral, ~12-23% baseline error),
**report `excess` conformity as the primary number for that model, not raw `CR`.** Excess
conformity subtracts each item's own unaided pull toward the distractor, so it stays valid even on
an uncalibrated bank — that is exactly the scenario `analyze.excess_conformity` was built for.

**Run this for at least three model families.** Two down (Qwen, Mistral) — both show the same
dissolution pattern with very strong significance. That is real cross-family evidence; a third
family makes it solid.

In [ ]:
from pathlib import Path

# Results go OUTSIDE the repo. The clone cell does `rm -rf` on REPO_DIR, so anything written
# inside it is destroyed the moment that cell is re-run — which is easy to do by accident after
# hours of GPU time. /kaggle/working also survives "Save Version".
OUT_DIR = Path("/kaggle/working/results_arms")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# google/gemma-2-9b-it is GATED and will 401 without an HF token + accepted licence (see the
# note above). microsoft/Phi-3.5-mini-instruct is ungated and gives a third, architecturally
# distinct family (different attention pattern, different training recipe) without the wait.
# Swap the Phi line for the Gemma line below once you have a token if you want Gemma specifically.
MODELS = [
    "Qwen/Qwen2.5-7B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "microsoft/Phi-3.5-mini-instruct",
    # "google/gemma-2-9b-it",  # gated -- needs HF_TOKEN + license acceptance, see note above
]

# --batch-size 1 (the default) = sequential = safe for paper numbers. See the note above.
for m in MODELS:
    print(f"\n\n{'#' * 92}\n#  {m}\n{'#' * 92}")
    !python {REPO_DIR}/scripts/run_arms.py --backend hf --model {m} --out-dir {OUT_DIR}

print("\nfiles written:")
!ls -la {OUT_DIR}

## Optional C — fast batched pass (exploratory only, NOT for the paper)

Same sweep, `--batch-size 16`. Use this only for an early directional look (e.g. checking a new
model isn't wildly off before committing hours of sequential time to it) — never as a source of
numbers you plan to report. Writes to a separate `_fastlook` directory so it can never be mistaken
for the sequential results above.

In [ ]:
FASTLOOK_DIR = Path("/kaggle/working/results_arms_fastlook")
FASTLOOK_DIR.mkdir(parents=True, exist_ok=True)

for m in MODELS:
    print(f"\n\n{'#' * 92}\n#  {m}  (FAST LOOK — batched, exploratory)\n{'#' * 92}")
    !python {REPO_DIR}/scripts/run_arms.py --backend hf --model {m} \
        --batch-size 16 --out-dir {FASTLOOK_DIR}